# GRIPL Experiments: Analysis of Dataset

#### Reading in data

In [51]:
import re
import pandas as pd

def load_fixed_dataframe(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        content = f.read()

    pattern = re.compile(r'^(\d+),', re.MULTILINE)
    matches = list(pattern.finditer(content))
    
    records = []

    for i in range(len(matches)):
        start = matches[i].start()
        end = matches[i+1].start() if i+1 < len(matches) else len(content)
        row_str = content[start:end].strip()
        
        first_comma = row_str.find(',')
        r_parts = row_str.rsplit(',', 4) 
        
        middle = r_parts[0][first_comma+1:].strip()
        
        xml_end_idx = middle.rfind('>') 
        json_start_idx = middle.find('[', xml_end_idx)
        
        if json_start_idx != -1:
            xml_content = middle[:json_start_idx].strip().strip(',')
            json_content = middle[json_start_idx:].strip()
        else:
            split_tag = "</bpmn:definitions>"
            split_idx = middle.find(split_tag)
            if split_idx != -1:
                xml_content = middle[:split_idx + len(split_tag)]
                json_content = middle[split_idx + len(split_tag):].strip().strip(',')
            else:
                xml_content = middle
                json_content = "[]"

        records.append({
            'id': row_str[:first_comma],
            'bpmn_xml': xml_content,
            'expected_values': json_content,
            'name': r_parts[-2].strip(),
            'dataset_id': r_parts[-1].strip()
        })

    return pd.DataFrame(records)

df = load_fixed_dataframe('evaluation_data.csv')

df.head()

print(f"# Testcases: {len(df)}")

# Check JSON content for critical activities
print("Critical Activities JSON (Vorschau):")
print(df['expected_values'].str[:50].tolist())

# Testcases: 93
Critical Activities JSON (Vorschau):
['[{"value": "sid-1386FB8B-EC49-4DF3-B330-8DFB706E7E', '[{"value": "sid-24E41551-0FD9-438B-B46C-CC2F70C778', '[{"value": "sid-37A4BE79-8627-4E33-A0A7-38FEA93FF3', '[{"value": "sid-6AF85299-5179-4780-B917-67C3678C00', '[{"value": "sid-33D863D3-01CA-4DED-9AB0-7D2095F7F1', '[{"value": "sid-85BA3A05-DB49-4E8A-AB3E-2FDF2898EB', '[{"value": "sid-23CF3684-BA3E-4729-8C67-ACE7FB17BC', '[]', '[{"value": "sid-DC8DB005-A84F-4BCB-B272-ED29C07A81', '[{"value": "sid-BF17F79E-2728-4C4E-A8F4-B95C0036CE', '[{"value": "sid-D1FED733-B5B7-454E-9761-777BAAAA78', '[{"value": "sid-58F5130E-A5E0-4E56-8138-A2FA45F137', '[{"value": "sid-D8309D0D-4482-428E-A648-46AE8CB714', '[{"value": "sid-4DF3FDE2-955E-4BEE-8227-F1E08D41A2', '[{"value": "c21b-1f26bc8df5723441a7c06602674717dc"', '[{"value": "sid-6BD08F19-8241-4F86-BFB7-157966A1FA', '[{"value": "sid-96D618DB-9481-472D-A470-FA1AF1C512', '[{"value": "sid-88FEC8E8-9132-46E9-89AB-7849E388A6', '[{"value": "sid-18CEB

In [52]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 93 entries, 0 to 92
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   id               93 non-null     str  
 1   bpmn_xml         93 non-null     str  
 2   expected_values  93 non-null     str  
 3   name             93 non-null     str  
 4   dataset_id       93 non-null     str  
dtypes: str(5)
memory usage: 3.8 KB


In [53]:
df.head()

,id,bpmn_xml,expected_values,name,dataset_id
0,4,"'<?xml version=""1.0"" encoding=""UTF-8""?><defini...","[{""value"": ""sid-1386FB8B-EC49-4DF3-B330-8DFB70...",Prepare surgical team,4
1,5,"'<?xml version=""1.0"" encoding=""UTF-8""?><defini...","[{""value"": ""sid-24E41551-0FD9-438B-B46C-CC2F70...",Prepare specific patient examination,4
2,6,"'<?xml version=""1.0"" encoding=""UTF-8""?><defini...","[{""value"": ""sid-37A4BE79-8627-4E33-A0A7-38FEA9...",Plan chemotherapy,4
3,98,"'<?xml version=""1.0"" encoding=""UTF-8""?>\n<defi...","[{""value"": ""sid-6AF85299-5179-4780-B917-67C367...",Pflege der Lehrendendaten,9
4,7,"'<?xml version=""1.0"" encoding=""UTF-8""?>\n<defi...","[{""value"": ""sid-33D863D3-01CA-4DED-9AB0-7D2095...",Performing diagnostic evaluation,4


#### Extract Activities

In [54]:
import ast

def get_clean_activities(df):
    all_rows = []
    
    # list of all BPMN Task types
    task_types = ['task', 'userTask', 'serviceTask', 'sendTask', 'receiveTask', 
                  'manualTask', 'businessRuleTask', 'scriptTask', 'subProcess', 'callActivity']
    # Regex for those tags, allowing for optional namespace prefix (e.g., bpmn:task)
    pattern = rf"<(?:[\w\d]+:)?({'|'.join(task_types)})\b[^>]*>"

    for _, row in df.iterrows():
        critical_ids = set()
        raw_expected = str(row.get('expected_values', '[]'))
        try:
            data_list = ast.literal_eval(raw_expected)
            critical_ids = {item['value'].strip().upper() for item in data_list if 'value' in item}
        except:
            found = re.findall(r'sid-[a-zA-Z0-9-]+', raw_expected)
            critical_ids = {id.strip().upper() for id in found}


        xml_content = str(row.get('bpmn_xml', ''))
        
        for match in re.finditer(pattern, xml_content, re.IGNORECASE):
            full_tag = match.group(0)
            
            id_match = re.search(r'id=["\'](.*?)["\']', full_tag)
            name_match = re.search(r'name=["\'](.*?)["\']', full_tag)
            
            akt_id = id_match.group(1) if id_match else ""
            
            all_rows.append({
                'dataset_id': row.get('dataset_id'),
                'prozess_id': row.get('id'),
                'prozess_name': row.get('name'),
                'Aktivitäts_id': akt_id,
                'aktivitätsname': name_match.group(1) if name_match else "",
                'label': 1 if akt_id.upper() in critical_ids else 0,
                'annotator_1': "",
                'annotator_1_comment': "",
                'annotator_2': "",
                'annotator_2_comment': "",
                'annotator_3': "",
                'annotator_3_comment': ""
            })

    return pd.DataFrame(all_rows)

df_activities = get_clean_activities(df)

# Check
print(f"Gesamt extrahiert: {len(df_activities)}")
print(f"Davon kritisch (Label 1): {df_activities['label'].sum()}")

# df_activities.to_excel("labelling_activities.xlsx", index=False)

Gesamt extrahiert: 940
Davon kritisch (Label 1): 554


In [55]:
df_activities.head(2)

,dataset_id,prozess_id,prozess_name,Aktivitäts_id,aktivitätsname,label,annotator_1,annotator_1_comment,annotator_2,annotator_2_comment,annotator_3,annotator_3_comment
0,4,4,Prepare surgical team,sid-1386FB8B-EC49-4DF3-B330-8DFB706E7ED4,Surgeon fills in form for histology,1,,,,,,
1,4,4,Prepare surgical team,sid-9F354A10-D1C4-48EC-87F7-824504FC653F,Wash disinfect hands,0,,,,,,


In [56]:
def get_stats_ultimate(xml_content):
    if isinstance(xml_content, bytes):
        xml_content = xml_content.decode('utf-8', errors='ignore')
    if not isinstance(xml_content, str) or len(xml_content) < 10:
        return None

    def count_tags(tag_list):
        total = 0
        for tag in tag_list:
            pattern = rf'<(?:[\w\d]+:)?{tag}[\s/>]'
            total += len(re.findall(pattern, xml_content))
        return total

    return {
        "Activities": count_tags(['task', 'userTask', 'serviceTask', 'sendTask', 'receiveTask', 
                                  'manualTask', 'businessRuleTask', 'scriptTask', 'subProcess', 'callActivity']),
        "Data objects": count_tags(['dataObject', 'dataObjectReference']),
        "Data association": count_tags(['dataInputAssociation', 'dataOutputAssociation']),
        "Events": count_tags(['startEvent', 'endEvent', 'intermediateCatchEvent', 'intermediateThrowEvent', 'boundaryEvent']),
        "Gateways": count_tags(['exclusiveGateway', 'parallelGateway', 'inclusiveGateway', 'eventBasedGateway', 'complexGateway']),
        "Pools": count_tags(['participant']),
        "Lanes": count_tags(['lane']),
        "Message flows": count_tags(['messageFlow']),
        "Annotations": count_tags(['textAnnotation'])
    }

# 2. Basis-Statistiken aus den XMLs extrahieren
all_stats_list = []
for index, row in df.iterrows():
    res = get_stats_ultimate(row['bpmn_xml'])
    if res:
        # IDs werden hier als gesäuberte Strings gespeichert
        res['prozess_id'] = str(row.get('id', '')).strip()
        res['dataset_id'] = str(row.get('dataset_id', '')).strip()
        res['prozess_name'] = row.get('name', f"Prozess_{index}")
        all_stats_list.append(res)

df_stats = pd.DataFrame(all_stats_list)

df_activities['prozess_id_clean'] = df_activities['prozess_id'].astype(str).str.strip()

critical_counts = df_activities[df_activities['label'] == 1].groupby('prozess_id_clean').size().to_dict()

df_stats['Activities (critical)'] = df_stats['prozess_id'].map(lambda x: critical_counts.get(x, 0))

cols = ["dataset_id", "prozess_id", "prozess_name", "Activities", "Activities (critical)", 
        "Data objects", "Data association", "Events", "Gateways", "Pools", "Lanes", "Annotations"]
df_stats = df_stats[[c for c in cols if c in df_stats.columns]]

display(df_stats.head())

# DEBUG-CHECK
if df_stats['Activities (critical)'].sum() == 0:
    print("\n⚠️ DEBUG INFO:")
    sample_id = df_stats['prozess_id'].iloc[0]
    print(f"Beispiel Prozess-ID in df_stats: '{sample_id}'")
    available_ids = list(critical_counts.keys())[:5]
    print(f"Verfügbare IDs in der Zählung: {available_ids}")

,dataset_id,prozess_id,prozess_name,Activities,Activities (critical),Data objects,Data association,Events,Gateways,Pools,Lanes,Annotations
0,4,4,Prepare surgical team,5,1,2,2,2,2,1,1,0
1,4,5,Prepare specific patient examination,4,3,4,5,5,4,2,3,0
2,4,6,Plan chemotherapy,4,4,6,4,5,2,1,2,0
3,9,98,Pflege der Lehrendendaten,8,8,0,0,2,9,1,2,7
4,4,7,Performing diagnostic evaluation,13,13,0,0,5,14,1,1,0


**Groups based on BPMN Model size**

We use the median = 10 for grouping.

In [57]:
df_stats.sort_values(by='Activities', ascending=False, inplace=True)

large_processes = df_stats.loc[df_stats['Activities'] >= 10, 'prozess_id']
small_processes = df_stats.loc[df_stats['Activities'] < 10, 'prozess_id']

print(f"Anzahl großer Prozesse (>=10 Aktivitäten): {len(large_processes)}")
print(f"Anzahl kleiner Prozesse (<10 Aktivitäten): {len(small_processes)}")

Anzahl großer Prozesse (>=10 Aktivitäten): 43
Anzahl kleiner Prozesse (<10 Aktivitäten): 50


In [58]:
# Group by dataset
dataset_summary = df_stats.groupby('dataset_id').agg({
    'prozess_id': 'count',                 # How many processes per dataset?
    'Activities': 'sum',                   # Total number of activities
    'Activities (critical)': 'sum',        # Total number of critical activities
    'Events': 'sum',
    'Gateways': 'sum',
    'Data objects': 'sum',
    'Pools': 'sum'
}).rename(columns={'prozess_id': 'Anzahl Prozesse'})

dataset_summary['Kritikalitäts-Quote (%)'] = (
    dataset_summary['Activities (critical)'] / dataset_summary['Activities'] * 100
).round(2)

dataset_summary['Ø Aktivitäten pro Prozess'] = (
    dataset_summary['Activities'] / dataset_summary['Anzahl Prozesse']
).round(1)


print("### SUMMARY BY DATASET ###")
display(dataset_summary)

dataset_summary.to_csv('dataset_summary_report_short.csv', sep=';', encoding='utf-8-sig')
print("File saved successfully!")

### SUMMARY BY DATASET ###


,Anzahl Prozesse,Activities,Activities (critical),Events,Gateways,Data objects,Pools,Kritikalitäts-Quote (%),Ø Aktivitäten pro Prozess
dataset_id,,,,,,,,,
4,27,195,148,146,126,73,41,75.90,7.2
5,10,92,39,54,78,12,10,42.39,9.2
6,7,84,49,76,31,6,18,58.33,12.0
7,13,165,25,64,69,9,20,15.15,12.7
8,9,126,68,71,72,28,15,53.97,14.0
9,27,278,225,174,202,9,41,80.94,10.3


File saved successfully!


In [59]:
import numpy as np

def get_stats_ultimate(xml_content):
    if isinstance(xml_content, bytes):
        xml_content = xml_content.decode('utf-8', errors='ignore')
    if not isinstance(xml_content, str) or len(xml_content) < 10:
        return None

    def count_tags(tag_list):
        total = 0
        for tag in tag_list:
            # Erkennt <messageFlow ...> oder <bpmn:messageFlow ...>
            pattern = rf'<(?:[\w\d]+:)?{tag}[\s/>]'
            total += len(re.findall(pattern, xml_content))
        return total

    return {
        "Activities": count_tags(['task', 'userTask', 'serviceTask', 'sendTask', 'receiveTask', 
                                  'manualTask', 'businessRuleTask', 'scriptTask', 'subProcess', 'callActivity']),
        "Activities (critical)": 0, 
        "Data objects": count_tags(['dataObject', 'dataObjectReference']),
        "Data association": count_tags(['dataInputAssociation', 'dataOutputAssociation']),
        "Events": count_tags(['startEvent', 'endEvent', 'intermediateCatchEvent', 'intermediateThrowEvent', 'boundaryEvent']),
        "Gateways": count_tags(['exclusiveGateway', 'parallelGateway', 'inclusiveGateway', 'eventBasedGateway', 'complexGateway']),
        "Pools": count_tags(['participant']),
        "Lanes": count_tags(['lane']),
        "Message flows": count_tags(['messageFlow']), 
        "Annotations": count_tags(['textAnnotation'])
    }

all_stats_list = []
for index, row in df.iterrows():
    res = get_stats_ultimate(row['bpmn_xml'])
    if res:
        res['prozess_id'] = str(row.get('id', '')).strip()
        res['dataset_id'] = str(row.get('dataset_id', '')).strip()
        all_stats_list.append(res)

df_stats = pd.DataFrame(all_stats_list)

# 3. KRITISCHE AKTIVITÄTEN MAPPEN
df_activities['prozess_id_clean'] = df_activities['prozess_id'].astype(str).str.strip()
critical_counts = df_activities[df_activities['label'] == 1].groupby('prozess_id_clean').size().to_dict()
df_stats['Activities (critical)'] = df_stats['prozess_id'].map(lambda x: critical_counts.get(x, 0))

metrics = [
    "Activities", "Activities (critical)", "Data objects", 
    "Data association", "Events", "Gateways", 
    "Pools", "Lanes", "Message flows", "Annotations"
]

summary_data = []

# Gruppierung nach Datensatz + Total-Berechnung
groups = list(df_stats.groupby('dataset_id'))
groups.append(('Total', df_stats)) # Gesamtauswertung anhängen

for name, group in groups:
    row = {"Category": name}
    row["Testcases total"] = len(group)
    row["# Activities"] = int(group["Activities"].sum())
    row["# Activities (critical)"] = int(group["Activities (critical)"].sum())
    
    # Ø +- SD für alle Metriken
    for m in metrics:
        mean = group[m].mean()
        std = group[m].std()
        row[f"Ø {m} ± SD"] = f"{mean:.2f} ± {np.nan_to_num(std):.2f}"
            
    summary_data.append(row)

final_table = pd.DataFrame(summary_data).set_index("Category").T

display(final_table)

final_table.to_csv('dataset_summary_report.csv', sep=';', encoding='utf-8-sig')
print("File saved successfully!")

Category,4,5,6,7,8,9,Total
Testcases total,27,10,7,13,9,27,93
# Activities,195,92,84,165,126,278,940
# Activities (critical),148,39,49,25,68,225,554
Ø Activities ± SD,7.22 ± 4.54,9.20 ± 5.59,12.00 ± 4.24,12.69 ± 10.99,14.00 ± 7.05,10.30 ± 5.28,10.11 ± 6.58
Ø Activities (critical) ± SD,5.48 ± 4.63,3.90 ± 3.57,7.00 ± 6.90,1.92 ± 2.93,7.56 ± 5.75,8.33 ± 5.46,5.96 ± 5.26
Ø Data objects ± SD,2.70 ± 3.09,1.20 ± 2.15,0.86 ± 2.27,0.69 ± 1.70,3.11 ± 3.33,0.33 ± 1.00,1.47 ± 2.51
Ø Data association ± SD,3.56 ± 3.91,1.60 ± 2.63,1.29 ± 3.40,1.23 ± 2.49,3.44 ± 4.33,0.41 ± 1.58,1.92 ± 3.27
Ø Events ± SD,5.41 ± 5.66,5.40 ± 7.82,10.86 ± 12.75,4.92 ± 5.59,7.89 ± 5.33,6.44 ± 6.08,6.29 ± 6.71
Ø Gateways ± SD,4.67 ± 5.41,7.80 ± 7.41,4.43 ± 4.50,5.31 ± 6.56,8.00 ± 7.09,7.48 ± 5.58,6.22 ± 5.99
Ø Pools ± SD,1.52 ± 1.12,1.00 ± 0.47,2.57 ± 1.13,1.54 ± 1.56,1.67 ± 1.32,1.52 ± 0.75,1.56 ± 1.10


File saved successfully!


## Result analysis

In [60]:
import json
from pathlib import Path
import pandas as pd
import numpy as np

folder_path = Path('..') 
files_to_load = list(folder_path.glob('*report*.json'))

print(f"Targeting files: {[f.name for f in files_to_load]}")

all_reports = []
for file_path in files_to_load:
    with open(file_path, 'r', encoding='utf-8-sig') as f:
        try:
            data = json.load(f)
            # Handle if the JSON file is a list [] or a single dictionary {}
            if isinstance(data, list):
                all_reports.extend(data)
            else:
                all_reports.append(data)
            print(f"✅ Loaded: {file_path.name}")
        except Exception as e:
            print(f"❌ Error loading {file_path.name}: {e}")

print(f"Total report objects in memory: {len(all_reports)}")

Targeting files: ['deepseek_report.json', 'google_report.json', 'mistral_report.json', 'openai_report.json']
✅ Loaded: deepseek_report.json
✅ Loaded: google_report.json
✅ Loaded: mistral_report.json
✅ Loaded: openai_report.json
Total report objects in memory: 4


In [61]:
# Adjust, if there is a specific model you want to exclude from the analysis
excluded_model = "DeepSeek-R1-Distill-Qwen-14B"

### RQ1.2 LLM size analysis

In [62]:
# ADJUST with models and buckets as needed
MODEL_BUCKETS = {
    "Gemini-2.5-Flash": "Frontier Efficient",
    "GPT-4o (2024-11-20)": "Frontier Efficient",
    "Mistral-Large-Instruct-2411": "Frontier Efficient",
    "DeepSeek-V3.1": "Frontier Efficient",
    "Gemma-3-12B": "Open Mid-Scale",
    "GPT-OSS-20B": "Open Mid-Scale",
    "Ministral-14B-2512": "Open Mid-Scale",
    "GPT-OSS-120B": "Large Open Frontier",
    "Mistral-Medium-3.1": "Large Open Frontier"
}

bucket_stats = {b: {str(r): {'tp': 0, 'fp': 0, 'fn': 0} for r in range(1, 6)} 
                for b in set(MODEL_BUCKETS.values())}

found_labels = set()

for report in all_reports:
    test_runs = report.get("testCasesByRun", {})
    for run_id, cases in test_runs.items():
        s_run_id = str(run_id)
        for tc in cases:
            label = tc.get("modelLabel", "").strip()
            found_labels.add(label) # For debugging
            
            if label == excluded_model or not label: 
                continue
            
            bucket = MODEL_BUCKETS.get(label)
            if bucket:
                bucket_stats[bucket][s_run_id]['tp'] += len(tc.get('correctActivityIds', []))
                bucket_stats[bucket][s_run_id]['fp'] += len(tc.get('falsePositiveIds', []))
                bucket_stats[bucket][s_run_id]['fn'] += len(tc.get('falseNegativeIds', []))

results = []
for bucket, runs in bucket_stats.items():
    t_tp = sum(r['tp'] for r in runs.values())
    t_fp = sum(r['fp'] for r in runs.values())
    t_fn = sum(r['fn'] for r in runs.values())
    
    # If no data at all for this bucket, we skip it
    if (t_tp + t_fp + t_fn) == 0: 
        continue

    p_micro = t_tp / (t_tp + t_fp) if (t_tp + t_fp) > 0 else 0
    r_micro = t_tp / (t_tp + t_fn) if (t_tp + t_fn) > 0 else 0
    f1_micro = (2 * p_micro * r_micro) / (p_micro + r_micro) if (p_micro + r_micro) > 0 else 0
    
    run_ps, run_rs, run_f1s = [], [], []
    for r_data in runs.values():
        tp, fp, fn = r_data['tp'], r_data['fp'], r_data['fn']
        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = (2 * p * r) / (p + r) if (p + r) > 0 else 0
        run_ps.append(p)
        run_rs.append(r)
        run_f1s.append(f1)
    
    results.append({
        "Bucket": bucket,
        "Prec": f"{p_micro:.3f} (±{np.std(run_ps):.3f})",
        "Rec": f"{r_micro:.3f} (±{np.std(run_rs):.3f})",
        "F1": f"{f1_micro:.3f} (±{np.std(run_f1s):.3f})"
    })


if not results:
    print("❌ ERROR: No results found. Found these labels in JSON:", found_labels)
else:
    df = pd.DataFrame(results, columns=["Bucket", "Prec", "Rec", "F1"])
    df = df.sort_values("Bucket")
    
    print("\n--- BUCKET PERFORMANCE WITH SD ---")
    print(df.to_string(index=False))

    # Save to CSV
    df.to_csv('1_2_llm_size_performance.csv', index=False, sep=';', encoding='utf-8-sig')
    print("\n✅ File saved successfully as '1_2_llm_size_performance.csv'")


--- BUCKET PERFORMANCE WITH SD ---
             Bucket           Prec            Rec             F1
 Frontier Efficient 0.802 (±0.007) 0.848 (±0.002) 0.824 (±0.004)
Large Open Frontier 0.816 (±0.013) 0.909 (±0.015) 0.860 (±0.008)
     Open Mid-Scale 0.764 (±0.005) 0.914 (±0.006) 0.832 (±0.002)

✅ File saved successfully as '1_2_llm_size_performance.csv'


#### BPMN-model based evaluation results

In [63]:
import numpy as np
from collections import defaultdict
import csv

In [64]:
def print_structure(data, indent=0):
    if isinstance(data, dict):
        for key, value in data.items():
            print("  " * indent + f"|- {key}")
            print_structure(value, indent + 1)
    elif isinstance(data, list) and len(data) > 0:
        # Just check the first item of a list to see the structure
        print("  " * indent + "|- [List Items]")
        print_structure(data[0], indent + 1)

# Run it on your first report
print_structure(all_reports[0])

|- metadata
  |- type
  |- modelLabels
    |- [List Items]
  |- modelTemperatures
    |- [List Items]
  |- modelTopPs
    |- [List Items]
  |- datasets
    |- [List Items]
      |- id
      |- name
  |- timestamp
  |- totalTestCases
  |- defaultEvaluationEndpoint
  |- seed
  |- totalRepetitions
  |- markdown
|- testCasesByRun
  |- 1
    |- [List Items]
      |- type
      |- testCaseId
      |- testCaseName
      |- datasetId
      |- imageSrc
      |- correctActivityIds
        |- [List Items]
      |- falsePositiveIds
      |- falseNegativeIds
        |- [List Items]
      |- expectedNamesWithIds
        |- [List Items]
      |- actualNamesWithIds
        |- [List Items]
      |- isSuccessful
      |- result
        |- [List Items]
          |- value
          |- reason
      |- amountOfRetries
      |- markdown
      |- modelLabel
  |- 2
    |- [List Items]
      |- type
      |- testCaseId
      |- testCaseName
      |- datasetId
      |- imageSrc
      |- correctActivityIds
      

In [65]:
test_case_history = defaultdict(list)

for report in all_reports:
    
    runs_container = report.get("testCasesByRun", {})
    
    # Iterate through runs -- ADJUST KEYS AS NEEDED
    for run_key in ["1", "2", "3", "4", "5"]:
        
        test_list = runs_container.get(run_key, [])
        
        for test_obj in test_list:
            tc_id = test_obj.get("testCaseId")
            
            if tc_id:
                test_obj['source_run_key'] = run_key        
                test_case_history[tc_id].append(test_obj)

print(f"✅ Grouped data for {len(test_case_history)} unique Test Case IDs.")

✅ Grouped data for 93 unique Test Case IDs.


In [66]:
scores_by_id = defaultdict(lambda: {"precision": [], "recall": [], "f1": [], "accuracy": []})

for tc_id, occurrences in test_case_history.items():
    for item in occurrences:
        
        if item.get("modelLabel") == excluded_model:
            continue
        
        tp = len(item.get('correctActivityIds', []))
        fp = len(item.get('falsePositiveIds', []))
        fn = len(item.get('falseNegativeIds', []))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        accuracy  = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0

        scores_by_id[tc_id]["precision"].append(precision)
        scores_by_id[tc_id]["recall"].append(recall)
        scores_by_id[tc_id]["f1"].append(f1)
        scores_by_id[tc_id]["accuracy"].append(accuracy)


print(f"{'TC_ID':<10} | {'Metric':<10} | {'Mean':<10} | {'Std Dev':<10}")
print("-" * 50)

for tc_id, metrics in scores_by_id.items():
    if not metrics["f1"]:
        continue
        
    print(f"Test Case: {tc_id}")
    for key in ["precision", "recall", "f1", "accuracy"]:
        data = metrics[key]
        mean = np.mean(data)
        std = np.std(data)
        print(f"{'':<10} | {key:<10} | {mean:>10.4f} | {std:>10.4f}")
    print("-" * 50)

TC_ID      | Metric     | Mean       | Std Dev   
--------------------------------------------------
Test Case: 7
           | precision  |     1.0000 |     0.0000
           | recall     |     0.4564 |     0.0192
           | f1         |     0.6265 |     0.0190
           | accuracy   |     0.4564 |     0.0192
--------------------------------------------------
Test Case: 4
           | precision  |     0.9667 |     0.1247
           | recall     |     1.0000 |     0.0000
           | f1         |     0.9778 |     0.0831
           | accuracy   |     0.9667 |     0.1247
--------------------------------------------------
Test Case: 6
           | precision  |     1.0000 |     0.0000
           | recall     |     0.7500 |     0.0000
           | f1         |     0.8571 |     0.0000
           | accuracy   |     0.7500 |     0.0000
--------------------------------------------------
Test Case: 10
           | precision  |     0.6970 |     0.1106
           | recall     |     0.9222 |     

In [67]:
filename = "1_2_testcase_analysis.csv"

with open(filename, mode='w', newline='') as file:
    writer = csv.writer(file)
    
    writer.writerow(["Test Case", "Metric", "Mean", "Std Dev"])
    
    for testcase, metrics in scores_by_id.items():
        for key in ["precision", "recall", "f1", "accuracy"]:
            data = metrics[key]
            if data:
                mean_val = np.mean(data)
                std_val = np.std(data)
                
                writer.writerow([testcase, key, f"{mean_val:.4f}", f"{std_val:.4f}"])

print(f"Successfully saved results to {filename}")

Successfully saved results to 1_2_testcase_analysis.csv


In [68]:
SIZE_THRESHOLD = 10  # Processes with < 10 total PII-relevant/total activities (Adjust as needed)

process_raw_scores = defaultdict(lambda: {"p": [], "r": [], "f1": [], "activity_count": 0})

for tc_id, occurrences in test_case_history.items():
    for item in occurrences:
        if item.get("modelLabel") == excluded_model:
            continue
        
        tp = len(item.get('correctActivityIds', []))
        fp = len(item.get('falsePositiveIds', []))
        fn = len(item.get('falseNegativeIds', []))
        
        total_relevant = tp + fn
        process_raw_scores[tc_id]["activity_count"] = total_relevant
        
        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0
        f = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
        
        process_raw_scores[tc_id]["p"].append(p)
        process_raw_scores[tc_id]["r"].append(r)
        process_raw_scores[tc_id]["f1"].append(f)


group_results = {
    "Small Processes": {"p": [], "r": [], "f1": []},
    "Large Processes": {"p": [], "r": [], "f1": []}
}

for tc_id, data in process_raw_scores.items():
    if not data["p"]: continue
    
    avg_p, avg_r, avg_f1 = np.mean(data["p"]), np.mean(data["r"]), np.mean(data["f1"])
    
    if data["activity_count"] < SIZE_THRESHOLD:
        tag = "Small Processes"
    else:
        tag = "Large Processes"
        
    group_results[tag]["p"].append(avg_p)
    group_results[tag]["r"].append(avg_r)
    group_results[tag]["f1"].append(avg_f1)

total_n = len(group_results["Small Processes"]["p"]) + len(group_results["Large Processes"]["p"])
print(f"VERIFICATION: Total Unique Test Cases Processed: {total_n}")
print(f"  - Small (<{SIZE_THRESHOLD} activities): {len(group_results['Small Processes']['p'])}")
print(f"  - Large (>={SIZE_THRESHOLD} activities): {len(group_results['Large Processes']['p'])}")
print("=" * 65)

print(f"{'Group':<18} | {'Metric':<10} | {'Macro-Mean':<10} | {'Process-SD':<10}")
print("-" * 65)

for group in ["Small Processes", "Large Processes"]:
    for m_key, label in [("p", "Precision"), ("r", "Recall"), ("f1", "F1-Score")]:
        vals = group_results[group][m_key]
        if vals:
            print(f"{group:<18} | {label:<10} | {np.mean(vals):>10.4f} | {np.std(vals):>10.4f}")
    print("-" * 65)

VERIFICATION: Total Unique Test Cases Processed: 93
  - Small (<10 activities): 68
  - Large (>=10 activities): 25
Group              | Metric     | Macro-Mean | Process-SD
-----------------------------------------------------------------
Small Processes    | Precision  |     0.7185 |     0.3514
Small Processes    | Recall     |     0.7814 |     0.3374
Small Processes    | F1-Score   |     0.7242 |     0.3336
-----------------------------------------------------------------
Large Processes    | Precision  |     0.8996 |     0.1155
Large Processes    | Recall     |     0.8595 |     0.1372
Large Processes    | F1-Score   |     0.8633 |     0.1066
-----------------------------------------------------------------


In [71]:
filename = "1_3_bpmn_size_analysis.csv"

with open(filename, mode='w', newline='') as file:
    writer = csv.writer(file)
    
    writer.writerow(["Group", "Metric", "Mean", "Std Dev", "Sample Size"])
    
    for group, metrics in group_results.items():
        for key in ["p", "r", "f1"]:
            data = metrics[key]
            if data:
                mean_val = np.mean(data)
                std_val = np.std(data)
                sample_size = len(data)
                
                # Write the row
                writer.writerow([group, key, f"{mean_val:.4f}", f"{std_val:.4f}", sample_size])

print(f"Successfully saved results to {filename}")

Successfully saved results to 1_3_bpmn_size_analysis.csv


In [72]:
critical_dataset_ids = [4, 9]

process_raw_scores = defaultdict(lambda: {"ds_id": None, "p": [], "r": [], "f1": []})

for tc_id, occurrences in test_case_history.items():
    for item in occurrences:
        if item.get("modelLabel") == excluded_model:
            continue
        
        ds_id = item.get("datasetId")
        tp, fp, fn = len(item.get('correctActivityIds', [])), len(item.get('falsePositiveIds', [])), len(item.get('falseNegativeIds', []))
        
        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0
        f = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
        
        process_raw_scores[tc_id]["ds_id"] = ds_id
        process_raw_scores[tc_id]["p"].append(p)
        process_raw_scores[tc_id]["r"].append(r)
        process_raw_scores[tc_id]["f1"].append(f)

domain_results = {
    "High-Criticality Domains": {"p": [], "r": [], "f1": []},
    "Other Domains": {"p": [], "r": [], "f1": []}
}

ds_id_counts = defaultdict(int)

for tc_id, data in process_raw_scores.items():
    if not data["p"]: continue
    
    avg_p, avg_r, avg_f1 = np.mean(data["p"]), np.mean(data["r"]), np.mean(data["f1"])
    
    target_domain = "High-Criticality Domains" if data["ds_id"] in critical_dataset_ids else "Other Domains"
    
    domain_results[target_domain]["p"].append(avg_p)
    domain_results[target_domain]["r"].append(avg_r)
    domain_results[target_domain]["f1"].append(avg_f1)
    
    ds_id_counts[data["ds_id"]] += 1

total_processed = sum(ds_id_counts.values())
print(f"VERIFICATION: Total Unique Test Cases Processed: {total_processed}")
print("Breakdown by datasetId:")
for ds_id, count in sorted(ds_id_counts.items()):
    label = "(High-Criticality)" if ds_id in critical_dataset_ids else "(Other)"
    print(f"  - Dataset {ds_id} {label}: {count} cases")
print("=" * 65)

print("MACRO-AVERAGE ANALYSIS BY DOMAIN")
print(f"{'Domain':<25} | {'Metric':<10} | {'Mean':<10} | {'SD (Process)':<10}")
print("-" * 65)

for domain, metrics in domain_results.items():
    count = len(metrics["p"])
    print(f"--- {domain} (N={count}) ---")
    for m_key, label in [("p", "Precision"), ("r", "Recall"), ("f1", "F1-Score")]:
        vals = metrics[m_key]
        if vals:
            print(f"{'':<25} | {label:<10} | {np.mean(vals):>10.4f} | {np.std(vals):>10.4f}")
    print("-" * 65)

VERIFICATION: Total Unique Test Cases Processed: 93
Breakdown by datasetId:
  - Dataset 4 (High-Criticality): 27 cases
  - Dataset 5 (Other): 10 cases
  - Dataset 6 (Other): 7 cases
  - Dataset 7 (Other): 13 cases
  - Dataset 8 (Other): 9 cases
  - Dataset 9 (High-Criticality): 27 cases
MACRO-AVERAGE ANALYSIS BY DOMAIN
Domain                    | Metric     | Mean       | SD (Process)
-----------------------------------------------------------------
--- High-Criticality Domains (N=54) ---
                          | Precision  |     0.8575 |     0.2314
                          | Recall     |     0.8716 |     0.2123
                          | F1-Score   |     0.8451 |     0.2108
-----------------------------------------------------------------
--- Other Domains (N=39) ---
                          | Precision  |     0.6422 |     0.3717
                          | Recall     |     0.7065 |     0.3678
                          | F1-Score   |     0.6461 |     0.3548
---------------------

In [ ]:
filename = "1_2_bpmn_domain_analysis.csv"

with open(filename, mode='w', newline='') as file:
    writer = csv.writer(file)
    
    writer.writerow(["Domain", "Metric", "Mean", "Std Dev", "Sample Size"])
    
    for domain, metrics in domain_results.items():
        for key in ["precision", "recall", "f1", "accuracy"]:
            data = metrics[key]
            if data:
                mean_val = np.mean(data)
                std_val = np.std(data)
                sample_size = len(data)
                
                writer.writerow([domain, key, f"{mean_val:.4f}", f"{std_val:.4f}", sample_size])

print(f"Successfully saved results to {filename}")

Successfully saved results to bpmn_domain_analysis.csv


In [73]:
english_processes = [3,4,5,6,7,8,9,10,12,13,16,17,18,19,20,21,25,26,51,62,114,116,118,119,33,49,34,36,38,41,63,66,106,123]
german_processes = [29,30,134,32,55,56,57,58,59,61,124,40,53,68,121,42,43,44,50,65,70,71,72,129,131,138,78,80,111,117,125,126,83,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,113,128,132,135,136,142,143]

print(f"✅ Number of English processes: {len(english_processes)}")
print(f"✅ Number of German processes: {len(german_processes)}")
print(f"✅ Total number of processes: {len(english_processes) + len(german_processes)}")

✅ Number of English processes: 34
✅ Number of German processes: 59
✅ Total number of processes: 93


In [74]:
process_raw_scores = defaultdict(lambda: {"p": [], "r": [], "f1": []})

for tc_id, occurrences in test_case_history.items():
    for item in occurrences:
        if item.get("modelLabel") == excluded_model:
            continue
        
        tp = len(item.get('correctActivityIds', []))
        fp = len(item.get('falsePositiveIds', []))
        fn = len(item.get('falseNegativeIds', []))
        
        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0
        f = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
        
        process_raw_scores[tc_id]["p"].append(p)
        process_raw_scores[tc_id]["r"].append(r)
        process_raw_scores[tc_id]["f1"].append(f)

# 2. Macro-Step: Aggregate into Language Buckets
lang_results = {"German": {"p": [], "r": [], "f1": []}, 
                "English": {"p": [], "r": [], "f1": []}}

for tc_id, data in process_raw_scores.items():
    if not data["p"]: continue # Skip if no valid runs found for this ID
    
    avg_p = np.mean(data["p"])
    avg_r = np.mean(data["r"])
    avg_f1 = np.mean(data["f1"])
    
    # Assigning based on your lists
    if tc_id in german_processes:
        tag = "German"
    elif tc_id in english_processes:
        tag = "English"
    else:
        continue # Ignore any IDs not explicitly mapped

    lang_results[tag]["p"].append(avg_p)
    lang_results[tag]["r"].append(avg_r)
    lang_results[tag]["f1"].append(avg_f1)

# 3. Final Formatted Output
print(f"Total Unique Processes Successfully Mapped: {len(lang_results['German']['p']) + len(lang_results['English']['p'])}")
print(f"Sample Sizes: German (N={len(lang_results['German']['p'])}), English (N={len(lang_results['English']['p'])})")
print("=" * 65)

for lang in ["German", "English"]:
    print(f"--- {lang} ---")
    for m_key, label in [("p", "Precision"), ("r", "Recall"), ("f1", "F1-Score")]:
        vals = lang_results[lang][m_key]
        if vals:
            print(f"{label:<10} | Mean: {np.mean(vals):.4f} | SD (Process): {np.std(vals):.4f}")
    print("-" * 65)

Total Unique Processes Successfully Mapped: 93
Sample Sizes: German (N=59), English (N=34)
--- German ---
Precision  | Mean: 0.6978 | SD (Process): 0.3520
Recall     | Mean: 0.7688 | SD (Process): 0.3378
F1-Score   | Mean: 0.7074 | SD (Process): 0.3339
-----------------------------------------------------------------
--- English ---
Precision  | Mean: 0.8876 | SD (Process): 0.1911
Recall     | Mean: 0.8605 | SD (Process): 0.2038
F1-Score   | Mean: 0.8557 | SD (Process): 0.1843
-----------------------------------------------------------------


In [75]:
filename = "1_2_bpmn_language_analysis.csv"

with open(filename, mode='w', newline='') as file:
    writer = csv.writer(file)
    
    writer.writerow(["Language", "Metric", "Mean", "Std Dev", "Sample Size"])
    
    for lang, metrics in lang_results.items():
        for key in ["p", "r", "f1"]:
            data = metrics[key]
            if data:
                mean_val = np.mean(data)
                std_val = np.std(data)
                sample_size = len(data)
                
                writer.writerow([lang, key, f"{mean_val:.4f}", f"{std_val:.4f}", sample_size])

print(f"✅ Successfully saved results to {filename}")

✅ Successfully saved results to 1_2_bpmn_language_analysis.csv
